In [ ]:
import sys; sys.path.append('..')
sys.path.append('../curved_linesearch/')
import MeshFEM, mesh, mesh_energy, benchmark, viewer, py_newton_optimizer
import differential_operators

import numpy as np
import igl

import matplotlib
from matplotlib import pyplot as plt

In [ ]:
import sim_utils, param_utils
import extra_utils, opt_utils

In [ ]:
from curved_linesearch import visualization

In [ ]:
import newton_flow
import newton_flow_utils as nfu

In [ ]:
from Benchmark import helper_funcs

# Newton Flow acts as Param

In [ ]:
model = 'Hilbert2.off'

In [ ]:
m = helper_funcs.read_mesh(f'../../../Models/TableOneModels/{model}')
print(f"Model: {model} Vertices: {m.numVertices()}")
print(f"Model: {model} Elements: {m.numElements()}")

In [ ]:
uv = mesh_energy.NodalVars(m, 2)
m_2d = mesh.Mesh('ToysMesh/Hilbert_init_2d.obj')
uv.setVars(m_2d.vertices().ravel())
nf = newton_flow.symmetric_dirichlet(m_2d, uv)

In [ ]:
# rest shapes are programmed by the 3d mesh vertices
nf.setRestVertexPositions(m.vertices())

In [ ]:
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, [nf])

## nf, prob, and opt settings

In [ ]:
nf.elementHessianShift = 1e-8
prob.hessianShift = 0
prob.useRelativeHessianShift = False

In [ ]:
FIX_VARS = False
always_project = False

In [ ]:
opt = prob.optimizer()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()
opt.options.niter = 200

In [ ]:
prob.energy()

## Test

In [ ]:
m_test = nf.mesh

In [ ]:
m_test.vertices()

In [ ]:
d = opt.newton_step()

In [ ]:
u_in_d_col = d.reshape(-1,2)[:,0]
v_in_d_col = d.reshape(-1,2)[:,1]

In [ ]:
u_grad = differential_operators.gradient(m_test, u_in_d_col)
v_grad = differential_operators.gradient(m_test, v_in_d_col)

In [ ]:
d_grad = np.stack((u_grad, v_grad), axis=1)   # (m, 2, 2)

In [ ]:
d_grad

In [ ]:
# brek

## RS 2D class Extrapolator 

In [ ]:
linear_extrapolator = extra_utils.LinearExtrapolator()

In [ ]:
RS_nf_extrapolator = extra_utils.RSNewtonFlowExtrapolation(prob)

In [ ]:
x0 = prob.getVars()
d0 = np.zeros_like(d)
RS_nf_extrapolator.linesearch_begin(x0, d0)
print(RS_nf_extrapolator.d_grad)

In [ ]:
alpha = 0
x_new = RS_nf_extrapolator.linesearch_eval(alpha)

In [ ]:
x_new

In [ ]:
prob.setVars(x_new.ravel())
print(prob.energy())

In [ ]:
brek

# Optimize

In [ ]:
# benchmark.reset()
# opt.optimize()
# benchmark.report()

In [ ]:
line_search_method = opt_utils.BruteForceLinesearch(alpha_step_size=0.01)

In [ ]:
benchmark.reset()
vertices_list = opt_utils.newton_extrapolate(opt, linear_extrapolator, line_search_method, max_iters=10, verbose=True)
benchmark.report()

In [ ]:
prob.energy()

In [ ]:
max_alpha = 10
line_search_method.alpha_step_size = 0.1
line_search_method.max_alpha = max_alpha

In [ ]:
benchmark.reset()
vertices_list = opt_utils.newton_extrapolate(opt, RS_nf_extrapolator, line_search_method, max_iters=100, verbose=True)
benchmark.report()